# 00 — Setup and your first crawl

Welcome! This is the first chapter of a hands-on tour of the `wcl_data`
Python package. By the end of this notebook you will have:

- the package installed in editable mode,
- a working `.env` file with fresh World Climbing credentials,
- an initialised SQLite warehouse at `data/wcl.sqlite`,
- thousands of seasons / events / competitions / athletes / results loaded into it.

If anything below errors out, fix it before moving on — the later notebooks
all read the warehouse this one populates.

## Prerequisites

- **Python 3.12+** (the package uses PEP 695 generic syntax; declared in `pyproject.toml`).
- **Internet access** — we'll hit the live World Climbing API.
- You've cloned the repo and this notebook is opening from inside it.

> All shell commands in this notebook use Jupyter's `!` prefix, which
> executes the line in a subprocess. The lines work the same in your terminal
> if you prefer.


## 1. Confirm the working directory

Every command below assumes the **repo root** is the current working
directory (so `.env` and `data/wcl.sqlite` resolve to the right place).
We `cd` up one level out of `notebooks/`.

In [ ]:
import os
from pathlib import Path

# notebooks/ -> repo root
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("cwd:", Path.cwd())
assert (Path.cwd() / "pyproject.toml").exists(), "Not in the repo root — fix this before continuing."


## 2. Install the package (editable, with the notebook extras)

`-e` installs the package in editable mode — edits under `src/wcl_data/`
take effect immediately without reinstalling. `[notebook]` pulls in
`jupyterlab` and `pandas`, which the later chapters use for nicer table
display.

This cell is idempotent; re-running it just confirms everything is
already installed.

In [ ]:
!pip install -e ".[notebook]"


## 3. Make sure `.env` exists

The package reads its configuration from a `.env` file at the repo root.
We copy from the committed `.env.example` if no `.env` is there yet — that
gives us a file with the right keys but empty credentials, which the
next cell will fill in.

In [ ]:
import shutil

env_path = Path(".env")
example_path = Path(".env.example")

if not env_path.exists():
    shutil.copy(example_path, env_path)
    print("Created .env from .env.example.")
else:
    print(".env already exists — leaving it alone.")

print()
print(env_path.read_text())


### What's in `.env`?

| Variable | Purpose |
|----------|---------|
| `WCL_CSRF_TOKEN` | Sent as the `X-Csrf-Token` HTTP header. The API rejects requests without it. |
| `WCL_SESSION_COOKIE` | Sent as the `Cookie` header. |
| `WCL_REFERER` | Sent as the `Referer` header. The World Climbing site checks this. |
| `WCL_MAX_WORKERS` | Concurrent HTTP workers (default `50`). |
| `WCL_CONNECT_TIMEOUT` | TCP/TLS connect timeout in seconds (default `5`). |
| `WCL_READ_TIMEOUT` | Per-request read timeout in seconds (default `120`). |
| `WCL_DB_PATH` | Where the SQLite warehouse lives. Relative paths resolve against the repo root. |
| `WCL_STALE_DAYS` | Default staleness threshold (rows older than this are re-fetched). |

The first two are empty right now — that's what the `auth` command in
the next step fixes.

## 4. Auto-fetch fresh credentials with `auth`

The World Climbing site uses Rails' standard CSRF + session-cookie pattern. The
CSRF token is published as a `<meta name="csrf-token" content="...">`
tag inside the homepage HTML, and the session cookie is set via
`Set-Cookie` on the same response — so a single plain `GET` is enough
to harvest both.

The `auth` subcommand does exactly that and writes the values back into
`.env` in place (other lines, comments and ordering are preserved).

> You'll want to re-run `auth` whenever `refresh` / `pull-new` / `hydrate`
> start failing with HTTP 401 or 403. In practice the session cookie
> lasts a few months.

In [ ]:
!python -m wcl_data auth


**Sanity check** — the two credential lines should now have values.

In [ ]:
for line in env_path.read_text().splitlines():
    if line.startswith("WCL_CSRF_TOKEN=") or line.startswith("WCL_SESSION_COOKIE="):
        # Don't print the full token. Just confirm it's non-empty.
        key, _, value = line.partition("=")
        print(f"{key} = {'(set, %d chars)' % len(value) if value else '(empty!)'}")


## 5. Initialise the SQLite schema (`init`)

This creates `data/wcl.sqlite` and applies the schema for all 15 tables
(9 base + 6 per-round, see ADR 0007) plus indexes. It is **idempotent** —
running it on an existing DB is a no-op, it never drops or migrates anything.

You only really need to run this once per machine, but it's safe to
run again whenever you're unsure.

In [ ]:
!python -m wcl_data init


## 6. Your first real crawl: `pull-new`

We now reach for `pull-new`, the everyday command. It works in two
passes:

1. **Force-discover** the *container* entities (seasons, season_leagues,
   events, competitions) by re-fetching them from the API regardless
   of staleness — so brand-new content gets picked up.
2. **Hydrate only the new athletes** discovered along the way (existing
   athlete profiles are skipped — they almost never change, and there
   are ~15k of them).

Why not just `refresh --stale-days 0`? That would also work, but it
re-fetches every athlete profile *and* every historical container's per-round
data, taking ~45-90 minutes. `pull-new` is ~3–5 minutes for the same
"catch what's new" job.

> **Heads up:** this cell streams its progress to the notebook output.
> It writes to disk continuously, so a Ctrl-C only loses the in-flight
> row, never a batch.

In [ ]:
!python -m wcl_data pull-new


## 7. Confirm what we just loaded (`status`)

`status` prints row counts and hydration coverage per table. No API
call — just a few `SELECT COUNT(*)` queries on the local DB.

In [ ]:
!python -m wcl_data status


### Reading the `status` output

For each *hydratable* table you see two numbers:

- **`rows`** — total rows present, including skeleton rows (discovered
  but not yet fully fetched).
- **`hydrated`** — rows where `last_fetched_at IS NOT NULL`, i.e. the
  full profile has been fetched.

Tables like `leagues`, `disciplines`, and `categories` show a dash
under `hydrated` because they don't carry a `last_fetched_at` column —
they're small static lookup tables that get populated as a side effect
of hydrating their parents.

If `rows` is non-zero but `hydrated` is much smaller, you have skeletons
waiting to be filled in — that's what `refresh` / `hydrate` are for.

## 8. What just happened, in plain English

The package walked the World Climbing API tree, working through five stages:

```
seasons → season_leagues → events → competitions → athletes
                                              ↓
                                          results
```

At each stage:

- It made concurrent HTTP requests (50 workers by default), streaming
  successes to disk as they arrived.
- 5xx, 429, and transport errors were retried with exponential backoff
  + jitter; `Retry-After` headers on 429s are honored.
- Other 4xx errors were treated as permanent (so a 404 on an athlete
  that doesn't exist doesn't waste retry budget).
- Five consecutive 401/403 across the worker pool would have aborted
  the run with `AuthFailureAbort` — protection against the silent-fail
  mode where every remaining row gets dropped to the WARN file log.
- Each successfully-fetched row was upserted into SQLite with its
  `last_fetched_at` timestamp.
- The `results` table was populated as a side effect of hydrating
  `competitions` (each competition's response includes the full ranking).

## What's next

Move on to [`01_the_data_model.ipynb`](01_the_data_model.ipynb) to
understand the schema you just populated.
